# List Fields

Suppose your database stores contractors, and each contractor has a variable-length list of skills. If your database supports array-valued columns, these are very likely stored in such a column rather than in a separate table with a foreign key to contractors.

Now suppose you want to filter for contractors who have two specific skills — will your semantic layer allow that? How about counting the contractors that have each particular skill? Or showing the complete list of skills available from contractors in each city?

SLayer types the field over such a column as `ARRAY` with an `element_type` (recognizing these automatically when ingesting schemas), and gives it semantics of its own:

- the **list functions** `contains`, `contains_any`, `contains_all` and `length` test a whole list;
- the list's **elements** are a child dataset `<field>.value` you can filter, group and aggregate, without fanning out the parent;
- `collect(x)` goes the other way and aggregates values *into* a list.

This notebook uses the two list fields of the Jaffle Shop demo: `products.ingredients` (the supplies a product is made from) and `orders.item_skus` (the SKU of each unit on an order, repeats kept).

**Prerequisites:** `pip install motley-slayer`

In [1]:
import os
import sys

import pandas as pd

pd.options.display.float_format = "{:,.2f}".format
pd.options.display.max_colwidth = 120
pd.options.display.width = 250

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", ".."))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "jaffle_data"))

from setup_jaffle import ensure_jaffle_shop

from slayer.core.errors import SlayerError
from slayer.core.models import field_type_text

engine, storage, datasets = ensure_jaffle_shop()


def run(query, **kwargs):
    return pd.DataFrame(engine.execute_sync(query, **kwargs).data)


def refused(query):
    try:
        engine.execute_sync(query)
    except SlayerError as error:
        print(error)

## The list type

Ingestion reads each database's native list type (`VARCHAR[]` here; BigQuery `REPEATED`, ClickHouse `Array(T)`, Snowflake `ARRAY`, …) as `datatype: ARRAY` with the element type it holds. The field's type text tells an agent what a list takes:

In [2]:
by_name = {d.name: d for d in datasets}
for dataset_name, field_name in [("products", "ingredients"), ("orders", "item_skus")]:
    field = by_name[dataset_name].get_field(field_name)
    print(f"{dataset_name}.{field_name}: {field_type_text(field)}")

products.ingredients: Array<String> (list: contains(ingredients, x), contains_any / contains_all(ingredients, [...]), length(ingredients) (distinct elements); it groups and compares as the set of its distinct elements; its elements are ingredients.value)
orders.item_skus: Array<String> (list: contains(item_skus, x), contains_any / contains_all(item_skus, [...]), length(item_skus) (distinct elements); it groups and compares as the set of its distinct elements; its elements are item_skus.value)


## Testing a whole list

`contains(list, x)` asks whether `x` is one of the list's elements; `contains_all` / `contains_any` compare it with another list, here an array literal; `length` counts the distinct elements. They work wherever an expression does — filters, computed dimensions, metrics.

In [3]:
run({
    "population": "products",
    "dimensions": ["name", {"field": "length(ingredients)", "alias": "ingredient_count"}],
    "filter": ["contains(ingredients, 'bread')"],
})

,products.name,products.ingredient_count
0,doctor stew,7
1,flame impala,9
2,mel-bun,8
3,nutellaphone who dis?,7
4,the krautback,9


In [4]:
run({
    "population": "products",
    "dimensions": ["name"],
    "filter": ["contains_all(ingredients, ['bread', 'cheese'])"],
})

,products.name
0,doctor stew
1,flame impala
2,mel-bun
3,the krautback


In [5]:
run({
    "population": "products",
    "dimensions": ["name"],
    "filter": ["contains_any(ingredients, ['banana', 'chai mix'])"],
})

,products.name
0,chai and mighty
1,nutellaphone who dis?


Absence is a `NOT contains(...)`: the products made without bread.

In [6]:
run({
    "population": "products",
    "dimensions": ["type"],
    "metrics": ["count(*)"],
    "filter": ["NOT contains(ingredients, 'bread')"],
})

,products.type,products._count
0,beverage,5


A list-valued `{variable}` in a list argument binds as a list, so a "basket" can come from the caller:

In [7]:
run(
    {"population": "orders", "metrics": ["count(*)"], "filter": ["contains_any(item_skus, {wanted})"]},
    variables={"wanted": ["JAF-004", "JAF-005"]},
)

,orders._count
0,75506


## A list as a set

A whole list groups and compares as the set of its distinct elements, shown sorted: products whose ingredients hold the same supplies, repeats and order aside, form one group. A list still has no single value to compare with a scalar or to sum, so those are refused with a typed error that names what to do instead. On a text field, `contains` is the substring test instead.

In [8]:
run({"population": "products", "dimensions": ["ingredients"], "metrics": ["count(*)"]})

,products.ingredients,products._count
0,"[bread, cheese, compostable cutlery - knife, cutlery - fork, ghost pepper sauce, napkin, pineapple, pulled pork, ser...",1
1,"[beef stew, bread, cheese, compostable cutlery - knife, cutlery - fork, napkin, serving boat]",1
2,"[16oz compostable clear cup, 16oz compostable clear lid, biodegradable straw, coffee, french vanilla syrup]",1
3,"[16oz compostable clear cup, 16oz compostable clear lid, biodegradable straw, mango, tangerine]",1
4,"[16oz compostable clear cup, 16oz compostable clear lid, biodegradable straw, chai mix, oatmilk, whey protein]",1
5,"[banana, bread, compostable cutlery - knife, cutlery - fork, napkin, nutella, serving boat]",1
6,"[16oz compostable clear cup, 16oz compostable clear lid, biodegradable straw, kiwi, lime]",1
7,"[16oz compostable clear cup, 16oz compostable clear lid, biodegradable straw, coffee]",1
8,"[bread, cheese, compostable cutlery - knife, cutlery - fork, house-pickled cabbage sauerkraut, lamb and pork bratwur...",1
9,"[bread, cheese, compostable cutlery - knife, cutlery - fork, melon, minced beef, napkin, serving boat]",1


In [9]:
refused({"population": "products", "dimensions": ["name"], "filter": ["ingredients = 'bread'"]})

ListTypeError: `ingredients` is a list and cannot be compared with a scalar by =.
  suggestion: Test membership with contains(ingredients, x), compare it with another list, or compare its elements through ingredients.value.


In [10]:
refused({"population": "orders", "metrics": ["sum(item_skus)"]})

ListTypeError: `item_skus` is a list: an aggregation over a list is only count or count_distinct; sum() cannot take it.
  suggestion: Aggregate its elements: sum(item_skus.value).


In [11]:
run({"population": "products", "dimensions": ["name"], "filter": ["contains(name, 'stew')"]})

,products.name
0,doctor stew


## Elements as rows

Every list field is also a one-to-many hop to its elements: `ingredients.value` is one row per non-NULL element, and `count(ingredients.*)` counts those rows. Group by the element, and `count(*)` is the number of **products** using each ingredient:

The warning is association's usual reminder: a product counts once under each of its ingredients, so the cells overlap and do not add up to the product total.

In [12]:
run({
    "population": "products",
    "dimensions": ["ingredients.value"],
    "metrics": ["count(*)"],
})

,products.ingredients.value,products._count
0,16oz compostable clear cup,5
1,16oz compostable clear lid,5
2,banana,1
3,beef stew,1
4,biodegradable straw,5
5,bread,5
6,chai mix,1
7,cheese,4
8,coffee,2
9,compostable cutlery - knife,5


On orders, `count(item_skus.*)` counts units while the associated `count(orders.*)` counts orders, so the two differ by the orders holding a SKU more than once:

In [13]:
run({
    "population": "orders",
    "dimensions": ["item_skus.value"],
    "metrics": [
        {"metric": "count(orders.*)", "alias": "orders_with_sku"},
        {"metric": "count(item_skus.*)", "alias": "units"},
    ],
})

,orders.item_skus.value,orders.orders_with_sku,orders.units
0,BEV-001,146729,152519
1,BEV-002,146312,152165
2,BEV-003,146539,152282
3,BEV-004,146806,152524
4,BEV-005,146183,151970
5,JAF-001,41446,45721
6,JAF-002,41553,45820
7,JAF-003,40969,45140
8,JAF-004,41174,45340
9,JAF-005,41073,45358


## No fan-out

Reading the elements never multiplies the parent rows: each aggregation runs over its own rows, so an order metric next to an element metric keeps its value. The order counts and revenue below are the same with or without the `units` metric beside them.

In [14]:
with_units = run({
    "population": "orders",
    "dimensions": ["stores.name"],
    "metrics": ["count(*)", "sum(order_total)", {"metric": "count(item_skus.*)", "alias": "units"}],
})
without_units = run({
    "population": "orders",
    "dimensions": ["stores.name"],
    "metrics": ["count(*)", "sum(order_total)"],
})
with_units

,orders.stores.name,orders._count,orders.order_total_sum,orders.units
0,Brooklyn,256173,2844560.84,384815
1,Chicago,109353,1177755.26,158226
2,New Orleans,15107,170489.92,23057
3,Philadelphia,193840,2152207.55,286765
4,San Francisco,92624,1025490.82,135976


In [15]:
with_units.drop(columns="orders.units").equals(without_units)

True

## Every element condition reads the same element

All `item_skus.value` conditions in one query constrain **one** element, so asking for a jaffle and a drink with two element filters matches nothing — no single SKU is both. To require several elements, use `contains_all`:

In [16]:
run({
    "population": "orders",
    "metrics": ["count(*)"],
    "filter": ["item_skus.value = 'JAF-001'", "item_skus.value = 'BEV-001'"],
})

,orders._count
0,0


In [17]:
run({
    "population": "orders",
    "metrics": ["count(*)"],
    "filter": ["contains_all(item_skus, ['JAF-001', 'BEV-001'])"],
})

,orders._count
0,12038


## Collecting values into a list

`collect(x)` is the aggregation that builds a list: the distinct non-NULL values of `x` in each cell, sorted. It takes fields, elements and expressions alike:

In [18]:
run({
    "population": "products",
    "dimensions": ["type"],
    "metrics": [{"metric": "collect(ingredients.value)", "alias": "all_ingredients"}],
})

,products.type,products.all_ingredients
0,beverage,"[16oz compostable clear cup, 16oz compostable clear lid, biodegradable straw, chai mix, coffee, french vanilla syrup..."
1,jaffle,"[banana, beef stew, bread, cheese, compostable cutlery - knife, cutlery - fork, ghost pepper sauce, house-pickled ca..."


`collect` works on regular fields too — here, the products of each type:

In [19]:
run({
    "population": "products",
    "dimensions": ["type"],
    "metrics": [{"metric": "collect(name)", "alias": "product_names"}],
})

,products.type,products.product_names
0,beverage,"[adele-ade, chai and mighty, for richer or pourover , tangaroo, vanilla ice]"
1,jaffle,"[doctor stew, flame impala, mel-bun, nutellaphone who dis?, the krautback]"


It also collects a field of a joined dataset, one list per parent row. Here are each customer's order statuses, fewest orders first:

In [20]:
run({
    "population": "customers",
    "dimensions": ["name"],
    "metrics": [
        {"metric": "count(orders.*)", "alias": "order_count"},
        {"metric": "collect(orders.status)", "alias": "statuses"},
    ],
    "order": [{"field": "order_count", "direction": "asc"}, {"field": "name"}],
    "limit": 5,
})

,customers.name,customers.order_count,customers.statuses
0,Marc Richardson,1,[completed]
1,Mrs. Emily Carter MD,1,[completed]
2,Ryan Davis,1,[completed]
3,Sarah Brown,1,[completed]
4,Carrie Foster,2,[completed]


A cell with no values gets the empty list, never `NULL` — here the orders that have no items:

In [21]:
run({
    "population": "orders",
    "dimensions": [{"field": "length(item_skus)", "alias": "item_count"}],
    "metrics": ["count(*)", {"metric": "collect(item_skus.value)", "alias": "skus"}],
    "filter": ["length(item_skus) < 2"],
})

,orders.item_count,orders._count,orders.skus
0,0,4567,[]
1,1,484928,"[BEV-001, BEV-002, BEV-003, BEV-004, BEV-005, JAF-001, JAF-002, JAF-003, JAF-004, JAF-005]"


## Lists across stages

A collected list stays a list in a later stage, with the same functions and elements. The first stage builds each customer's basket of SKUs ever ordered; the second counts the customers whose basket holds `JAF-005`, and how many customers ever bought each SKU:

In [22]:
baskets = {
    "name": "baskets",
    "population": "orders",
    "dimensions": ["customer_id"],
    "metrics": [{"metric": "collect(item_skus.value)", "alias": "skus"}],
}
run([baskets, {"population": "baskets", "metrics": ["count(*)"], "filter": ["contains(skus, 'JAF-005')"]}])

,baskets._count
0,1628


In [23]:
run([
    baskets,
    {"population": "baskets", "dimensions": ["skus.value"], "metrics": ["count(*)"]},
])

,baskets.skus.value,baskets._count
0,BEV-001,2570
1,BEV-002,2573
2,BEV-003,2571
3,BEV-004,2573
4,BEV-005,2573
5,JAF-001,1632
6,JAF-002,1629
7,JAF-003,1629
8,JAF-004,1632
9,JAF-005,1628


## Where lists run

[Database support](../../database-support.md) names the databases that run lists, the list functions and `collect`. See [List fields](../../concepts/queries.md#list-fields) and [Lists](../../concepts/references.md#lists) for the full rules.